# Uso de script de modularización — bielección (bloque largo t-2 -> t)

Misma estructura que `ventana_t-1/01.1_lasso_voto_valido.ipynb`, pero sobre
`data/tfi_data/panel_ventanas_bieleccion.csv` (`ml_models.construir_panel_lasso_bieleccion`,
D14 -- una fila por transición bielección, features de trayectoria
trimestral `_nivel_trim`/`_pendiente_trim`/`_volatilidad_trim`/`_final_trim`
calculadas sobre las filas `tipo_fila=="trimestre"` de
`panel/t-2/panel_bieleccion_trimestral_<nivel>.csv`, en vez de la serie
mensual). Variable a predecir: `delta_v` = `share_oficialismo` de la fila
`eleccion_t` menos la de `eleccion_t_menos_2` (bloque largo, no la ventana
corta t-1 -> t de `01.1_lasso_voto_valido.ipynb`).

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import pandas as pd

general_path = "/workspaces/analisis-politica-economia/"
data_path = f"{general_path}data/tfi_data/"
sys.path.insert(0, f"{general_path}/src")
from ml_models.cargar_panel import cargar_panel, columnas_candidatas
from ml_models.lasso import *
NIVELES = ["municipal", "provincial", "nacional"]
paneles = {nivel: cargar_panel(nivel, f"{data_path}panel_ventanas_bieleccion.csv") for nivel in NIVELES}

for nivel, df in paneles.items():
    print(f"{nivel}: {df.shape[0]} filas x {df.shape[1]} columnas")

municipal: 11 filas x 82 columnas
provincial: 11 filas x 82 columnas
nacional: 11 filas x 82 columnas


In [3]:
cols_vc_por_nivel = {}
corr_por_nivel = {}
for nivel in NIVELES:
    df = paneles[nivel]
    cols_vc = columnas_candidatas(df, excluir_adicional=["delta_v"])

    corr = df[cols_vc].corr(method="pearson")  # pairwise, ignora NaN automáticamente
    cols_vc_por_nivel[nivel] = cols_vc
    corr_por_nivel[nivel] = corr
for nivel in NIVELES:
    print(f"{nivel}: {len(cols_vc_por_nivel[nivel])} variables candidatas, N={len(paneles[nivel])}")

municipal: 75 variables candidatas, N=11
provincial: 75 variables candidatas, N=11
nacional: 75 variables candidatas, N=11


### Paso 2 — Sub-selección: colapsar clusters redundantes

Mismo criterio que `01.1_lasso_voto_valido.ipynb` (umbral `|r| >= 0.90`,
single-linkage, desempate por sufijo `_nivel_trim` preferido sobre
`_final_trim`/`_pendiente_trim`/`_volatilidad_trim`, después por
prioridad teórica de la variable).

In [4]:
UMBRAL_REDUNDANCIA = 0.90
PRIORIDAD_TEORICA = ["ipc", "desocupacion", "icg", "icc", "salario_real", "tc_oficial", "reservas", "resultado_fiscal", "emae"]
ORDEN_SUFIJO = ["_nivel_trim", "_final_trim", "_pendiente_trim", "_volatilidad_trim"]

clusters_por_nivel = {}
columnas_finales_por_nivel = {}

for nivel in NIVELES:
    df = paneles[nivel]
    cols_vc = cols_vc_por_nivel[nivel]
    corr = corr_por_nivel[nivel]
    print(f"\n\nNivel: {nivel}")
    clusters_por_nivel[nivel] = encontrar_redundantes(corr, UMBRAL_REDUNDANCIA)
    columnas_finales_por_nivel[nivel] = [elegir_representante(cl, df, ORDEN_SUFIJO, PRIORIDAD_TEORICA) if len(cl) > 1 else next(iter(cl)) for cl in clusters_por_nivel[nivel]]

    print(f"De {len(cols_vc)} columnas candidatas, quedan {len(columnas_finales_por_nivel[nivel])} tras colapsar clusters (umbral={UMBRAL_REDUNDANCIA})\n")
    for cl in clusters_por_nivel[nivel]:
        if len(cl) > 1:
            elegido = elegir_representante(cl, df, ORDEN_SUFIJO, PRIORIDAD_TEORICA)
            print(f"cluster ({len(cl)}): {sorted(cl)} -> queda: {elegido}")



Nivel: municipal
De 75 columnas candidatas, quedan 56 tras colapsar clusters (umbral=0.9)

cluster (3): ['desocupacion_nivel_trim', 'emae_final_trim', 'emae_nivel_trim'] -> queda: desocupacion_nivel_trim
cluster (6): ['hacinamiento_medio_cobertura_parcial', 'pct_hogares_ayuda_social_gobierno_cobertura_parcial', 'pct_hogares_prestamo_bancario_cobertura_parcial', 'pct_hogares_vendio_pertenencias_cobertura_parcial', 'pct_sin_cobertura_salud_cobertura_parcial', 'tasa_informalidad_cobertura_parcial'] -> queda: hacinamiento_medio_cobertura_parcial
cluster (3): ['ipc_cobertura_parcial', 'resultado_fiscal_cobertura_parcial', 'salario_real_cobertura_parcial'] -> queda: ipc_cobertura_parcial
cluster (3): ['pct_hogares_ayuda_social_gobierno_final_trim', 'pct_hogares_ayuda_social_gobierno_nivel_trim', 'resultado_fiscal_volatilidad_trim'] -> queda: pct_hogares_ayuda_social_gobierno_nivel_trim
cluster (2): ['pct_hogares_prestamo_bancario_volatilidad_trim', 'pct_hogares_vendio_pertenencias_volatili

In [5]:
datos_final = {}
for nivel in NIVELES:
    X, y = construir_Xy_final(nivel, columnas_finales_por_nivel[nivel], paneles, target="delta_v")
    datos_final[nivel] = (X, y)
    print(f"{nivel}: N={len(y)}, P={X.shape[1]}")

[municipal] excluye columna(s) sin varianza: ['icc_cobertura_parcial', 'icg_cobertura_parcial', 'tc_oficial_cobertura_parcial']
municipal: N=11, P=53
[provincial] excluye columna(s) sin varianza: ['icc_cobertura_parcial', 'icg_cobertura_parcial', 'tc_oficial_cobertura_parcial']
provincial: N=11, P=53
[nacional] excluye columna(s) sin varianza: ['icc_cobertura_parcial', 'icg_cobertura_parcial', 'tc_oficial_cobertura_parcial']
nacional: N=11, P=53


### Paso 3 — LASSO por coordinate descent (implementación propia)

Formulación: `(1/2n)·‖y - Xβ‖² + α·‖β‖₁` (convención sklearn/glmnet). `X`
estandarizada a mano (`ddof=0`), `y` centrada; intercepto = `media(y)`, no
se penaliza. Mismo chequeo KKT/OLS que `01.1_lasso_voto_valido.ipynb`.

In [6]:
for nivel in NIVELES:
    X_df, y_ser = datos_final[nivel]
    X_std, medias, desvios = estandarizar(X_df)
    y_centrado = y_ser.values - y_ser.mean()
    n = len(y_centrado)

    alpha_prueba = 1.0
    beta_manual = lasso_coordinate_descent(X_std, y_centrado, alpha_prueba)

    kkt = verificar_kkt(X_std, y_centrado, beta_manual, alpha_prueba, n)
    print(f"{nivel} - KKT:", kkt)

    beta_alpha_cero = lasso_coordinate_descent(X_std, y_centrado, alpha=0.0, max_iter=5000)
    beta_ols, *_ = np.linalg.lstsq(X_std, y_centrado, rcond=None)
    print(f"{nivel} - Máxima diferencia vs. OLS (alpha=0):", np.max(np.abs(beta_alpha_cero - beta_ols)))

municipal - KKT: {'error_max_en_activos': np.float64(8.56679033933716e-07), 'exceso_max_en_inactivos': np.float64(-0.00704642213563933), 'n_activos': np.int64(9)}
municipal - Máxima diferencia vs. OLS (alpha=0): 7.128333889651094
provincial - KKT: {'error_max_en_activos': np.float64(6.719236060881428e-07), 'exceso_max_en_inactivos': np.float64(-0.0011141499135028932), 'n_activos': np.int64(9)}
provincial - Máxima diferencia vs. OLS (alpha=0): 4.547765482715289
nacional - KKT: {'error_max_en_activos': np.float64(9.684102150187002e-07), 'exceso_max_en_inactivos': np.float64(-0.03529910109518575), 'n_activos': np.int64(10)}
nacional - Máxima diferencia vs. OLS (alpha=0): 7.6097425015296825


### Paso 4 — Grilla de alpha + LOO-CV manual

In [7]:
resultados_cv = {}
for nivel in NIVELES:
    X_df, y_ser = datos_final[nivel]
    resultados_cv[nivel] = lasso_loocv_manual(X_df, y_ser, factor_extension=3.0)
    r = resultados_cv[nivel]
    print(f"{nivel}: alpha_min={r['alpha_min']:.4f}  alpha_1se={r['alpha_1se']:.4f}  (techo grilla={r['alphas'][-1]:.4f})")

municipal: alpha_min=11.7054  alpha_1se=31.4019  (techo grilla=31.4019)
provincial: alpha_min=10.4026  alpha_1se=24.2375  (techo grilla=24.2375)
nacional: alpha_min=10.4133  alpha_1se=27.9356  (techo grilla=27.9356)


In [8]:
for nivel in NIVELES:
    X_df, y_ser = datos_final[nivel]
    print(f"--- {nivel} ---")
    print(verificar_saturacion(X_df, y_ser, factores=[1, 3, 10]))
    print()

--- municipal ---
   factor_extension       techo  alpha_min   alpha_1se     mse_min  \
0                 1   10.467302  10.467302   10.467302  287.935475   
1                 3   31.401907  11.705355   31.401907  279.189544   
2                10  104.673022  12.631863  104.673022  279.737552   

   mse_en_techo  
0    287.935475  
1    279.737552  
2    279.737552  

--- provincial ---
   factor_extension      techo  alpha_min  alpha_1se     mse_min  mse_en_techo
0                 1   8.079176   8.079176   8.079176  151.465447    151.465447
1                 3  24.237529  10.402588  24.237529  148.863020    148.863020
2                10  80.791763   9.749891  80.791763  148.863020    148.863020

--- nacional ---
   factor_extension      techo  alpha_min  alpha_1se     mse_min  mse_en_techo
0                 1   9.311863   9.311863   9.311863  205.718002    205.718002
1                 3  27.935590  10.413253  27.935590  203.338428    203.610815
2                10  93.118635  11.237

### Paso 5 — Ajuste final: coeficientes y mejora sobre baseline trivial

`baseline_trivial_loocv`: MSE en LOO de predecir el promedio de los demás
puntos. `mse_en_alpha`: mismo esquema de LOO, para un alpha puntual.

In [9]:
resumen = []
coeficientes_min, coeficientes_1se = {}, {}

for nivel in NIVELES:
    X_df, y_ser = datos_final[nivel]
    r = resultados_cv[nivel]

    base = baseline_trivial_loocv(y_ser)
    mse_min = mse_en_alpha(X_df, y_ser, r["alpha_min"])
    mse_1se = mse_en_alpha(X_df, y_ser, r["alpha_1se"])

    resumen.append({
        "nivel": nivel,
        "baseline_mse": base,
        "mejora_alpha_min_%": 100 * (1 - mse_min / base),
        "mejora_alpha_1se_%": 100 * (1 - mse_1se / base),
    })

    coeficientes_min[nivel] = ajustar_final(X_df, y_ser, r["alpha_min"])
    coeficientes_1se[nivel] = ajustar_final(X_df, y_ser, r["alpha_1se"])

tabla_resumen = pd.DataFrame(resumen).set_index("nivel")
print(tabla_resumen)

            baseline_mse  mejora_alpha_min_%  mejora_alpha_1se_%
nivel                                                           
municipal     279.737552            0.195901                 0.0
provincial    148.863020            0.000000                 0.0
nacional      203.610815            0.133779                 0.0


In [10]:
tabla_coef_min = pd.DataFrame(coeficientes_min)
tabla_coef_min = tabla_coef_min[(tabla_coef_min != 0).any(axis=1)]
print("Coeficientes distintos de cero (alpha_min):")
tabla_coef_min

Coeficientes distintos de cero (alpha_min):


,municipal,provincial,nacional
emae_nivel_trim,NaN,NaN,0.0
resultado_fiscal_pendiente_trim,0.0,0.0,NaN


### Paso 6 — Chequeo de estabilidad (leave-one-transition-out), los tres niveles

In [11]:
for nivel in NIVELES:
    X_df, y_ser = datos_final[nivel]
    alpha = resultados_cv[nivel]["alpha_1se"]
    resultado = estabilidad_seleccion(nivel, alpha, paneles[nivel], columnas_finales_por_nivel[nivel], "delta_v", X_df, y_ser)
    sobrevivientes = resultado.loc[:, (resultado != 0).any(axis=0)]

    print(f"--- {nivel} (alpha_1se={alpha:.3f}) ---")
    if sobrevivientes.empty:
        print("Ninguna variable sobrevive en ninguna de las corridas leave-one-transition-out.\n")
    else:
        print(sobrevivientes)
        print()

--- municipal (alpha_1se=31.402) ---
Ninguna variable sobrevive en ninguna de las corridas leave-one-transition-out.

--- provincial (alpha_1se=24.238) ---
Ninguna variable sobrevive en ninguna de las corridas leave-one-transition-out.

--- nacional (alpha_1se=27.936) ---
Ninguna variable sobrevive en ninguna de las corridas leave-one-transition-out.

